In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
enrolment = pd.read_csv('cleaned dataset/api_data_aadhar_enrolment.csv')
demographic = pd.read_csv('cleaned dataset/api_data_aadhar_demographic.csv')
biometric = pd.read_csv('cleaned dataset/api_data_aadhar_biometric.csv')

In [ ]:
election_calendar = {
    'delhi': '2025-02-05',
    'bihar': '2025-11-06',
    'gujarat': '2025-06-19',
    'kerala': '2025-06-19',
    'west bengal': '2025-06-19'
}

election_state = list(election_calendar.keys())
analysis_window = 90 # Look at 90 days before

In [ ]:
def election_surge(election_date):
   

In [ ]:
--- 3. Analysis Function: The "Surge Detector" ---
def analyze_election_surge(state_name, election_date_str):
    election_date = pd.to_datetime(election_date_str)
    
    # Define the "Danger Window" (90 days before election)
    start_window = election_date - pd.Timedelta(days=90)
    end_window = election_date - pd.Timedelta(days=10) # 10 days before poll (lists usually freeze)
    
    # Normal Baseline (Rest of the year outside this window)
    # We use this to see if the surge is "abnormal"
    
    # --- A. Analyze New Enrollments (Fake Voter Risk) ---
    # Filter for State and Adults (Age > 17)
    mask_state = df_enrol['state'].str.lower() == state_name.lower()
    mask_adult = df_enrol['age 17 and above'] > 0 # Assuming this column tracks adults
    
    state_enrol = df_enrol[mask_state & mask_adult]
    
    # Split into "Election Window" vs "Normal Time"
    surge_enrol = state_enrol[(state_enrol['date'] >= start_window) & (state_enrol['date'] <= end_window)]
    normal_enrol = state_enrol[~((state_enrol['date'] >= start_window) & (state_enrol['date'] <= end_window))]
    
    # Calculate Daily Averages
    avg_daily_new_voters = surge_enrol['age 17 and above'].sum() / 80 # approx 80 days
    baseline_daily_new = normal_enrol['age 17 and above'].sum() / (365 - 80) if len(normal_enrol) > 0 else 1
    
    # --- B. Analyze Demographic Updates (Moved Voters Risk) ---
    mask_demo_state = df_demo['state'].str.lower() == state_name.lower()
    state_demo = df_demo[mask_demo_state]
    
    surge_demo = state_demo[(state_demo['date'] >= start_window) & (state_demo['date'] <= end_window)]
    
    total_new_adults = surge_enrol['age 17 and above'].sum()
    total_updates = surge_demo['age 17 and above'].sum() if 'age 17 and above' in df_demo.columns else 0
    
    return {
        'State': state_name.title(),
        'Election_Date': election_date_str,
        'New_Adult_Aadhaars': int(total_new_adults),
        'Demographic_Updates': int(total_updates),
        'Daily_Avg_During_Surge': round(avg_daily_new_voters, 1),
        'Normal_Daily_Baseline': round(baseline_daily_new, 1),
        'Surge_Multiplier': round(avg_daily_new_voters / baseline_daily_new, 2)
    }

In [ ]:
# 2. Data Preparation
df = demographic.copy()
df['date'] = pd.to_datetime(df['date'])
target_date = pd.to_datetime(election_date)

In [ ]:
# Filter data to the relevant window (90 days before to 30 days after)
start_date = target_date - pd.Timedelta(days=analysis_window)
end_date = target_date + pd.Timedelta(days=30)

mask = (df['date'] >= start_date) & (df['date'] <= end_date)
election_df = df.loc[mask].groupby('date')['age 17 and above'].sum().reset_index()

In [ ]:
# 3. Visualization
plt.figure(figsize=(12, 6))

# Plot the Trend
plt.plot(election_df['date'], election_df['age 17 and above'], color='blue', linewidth=2)

# Mark the Election Day
plt.axvline(target_date, color='red', linewidth=3, linestyle='--', label='Election Day')

# Highlight the "Panic Window" (e.g., 30 days before)
panic_start = target_date - pd.Timedelta(days=30)
plt.axvspan(panic_start, target_date, color='red', alpha=0.1, label='Pre-Election Rush')

plt.title(f'Impact of Elections on Adult (18+) Updates: {election_date}', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Daily Adult Updates')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:


# 4. Calculate the "Panic Factor"
# Compare "30 Days Before" vs "60-90 Days Before" (Normal Baseline)
rush_period = election_df[(election_df['date'] >= panic_start) & (election_df['date'] < target_date)]
baseline_period = election_df[election_df['date'] < panic_start]

avg_rush = rush_period['age 17 and above'].mean()
avg_base = baseline_period['age 17 and above'].mean()

surge_pct = ((avg_rush - avg_base) / avg_base) * 100

print(f"--- Election Surge Analysis ---")
print(f"Baseline Daily Updates: {avg_base:.0f}")
print(f"Pre-Election Daily Updates: {avg_rush:.0f}")
print(f"Panic Surge Factor: +{surge_pct:.2f}%")